In [60]:
%load_ext autoreload
%autoreload 2
%reset -f

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [61]:
from pathlib import Path
import os
from os.path import join
import sys

# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers
from locallib.pandas import *

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.KPIHubConnection import *
from lib.query.bank import *

from datetime import date
from datetime import timedelta

In [ ]:
customer_name = 'Cadent'
customer_id = Query(query = f"SELECT CustomerId FROM KPI_Customer WHERE Name = '{customer_name}'").execute([KPIHub_Conn]).values[0][0]
tableList = [KPI_PeakAboveSAT]
aggregator = ['ReportYear','WeekNumber','BoundaryRegion']
data = {}
for table in tableList:
    data[table] = Query(query = f"SELECT * FROM {table.name} WHERE CustomerId IN (SELECT CustomerId FROM KPI_Customer WHERE Name = '{customer_name}')").execute([KPIHub_Conn])


In [63]:

r = data[KPI_PeakAboveSAT].groupby(aggregator).agg(PeakAboveSATCount=('PeakId', 'count')).reset_index()
r

,ReportYear,WeekNumber,BoundaryRegion,PeakAboveSATCount
0,2026,2,North London,3
1,2026,5,North West,17
2,2026,6,East Midlands,6
3,2026,6,North London,7
4,2026,6,North West,4
...,...,...,...,...
98,2026,27,West Midlands,3
99,2026,28,East of England,1
100,2026,28,North London,4
101,2026,28,North West,9


In [64]:
r_long = r.melt(id_vars=aggregator, var_name='KPIId', value_name='Value')
r_long['Id'] = r_long.apply(lambda row: f"{row['KPIId']}_{customer_name}_Y{row['ReportYear']}_W{row['WeekNumber']}", axis=1)
r_long = r_long.rename(columns={'WeekNumber': 'PeriodValue', 'ReportYear': 'Year'})
r_long['PeriodType'] = "Weekly"
r_long['LastUpdated'] = datetime.now()
r_long['CustomerId'] = customer_id


In [65]:
r_long

,Year,PeriodValue,BoundaryRegion,KPIId,Value,Id,PeriodType,LastUpdated,CustomerId
0,2026,2,North London,PeakAboveSATCount,3,PeakAboveSATCount_Cadent_Y2026_W2,Weekly,2026-07-08 03:56:30.009200,BD4D080B-1D12-D329-ABD0-39FEB9804E98
1,2026,5,North West,PeakAboveSATCount,17,PeakAboveSATCount_Cadent_Y2026_W5,Weekly,2026-07-08 03:56:30.009200,BD4D080B-1D12-D329-ABD0-39FEB9804E98
2,2026,6,East Midlands,PeakAboveSATCount,6,PeakAboveSATCount_Cadent_Y2026_W6,Weekly,2026-07-08 03:56:30.009200,BD4D080B-1D12-D329-ABD0-39FEB9804E98
3,2026,6,North London,PeakAboveSATCount,7,PeakAboveSATCount_Cadent_Y2026_W6,Weekly,2026-07-08 03:56:30.009200,BD4D080B-1D12-D329-ABD0-39FEB9804E98
4,2026,6,North West,PeakAboveSATCount,4,PeakAboveSATCount_Cadent_Y2026_W6,Weekly,2026-07-08 03:56:30.009200,BD4D080B-1D12-D329-ABD0-39FEB9804E98
...,...,...,...,...,...,...,...,...,...
98,2026,27,West Midlands,PeakAboveSATCount,3,PeakAboveSATCount_Cadent_Y2026_W27,Weekly,2026-07-08 03:56:30.009200,BD4D080B-1D12-D329-ABD0-39FEB9804E98
99,2026,28,East of England,PeakAboveSATCount,1,PeakAboveSATCount_Cadent_Y2026_W28,Weekly,2026-07-08 03:56:30.009200,BD4D080B-1D12-D329-ABD0-39FEB9804E98
100,2026,28,North London,PeakAboveSATCount,4,PeakAboveSATCount_Cadent_Y2026_W28,Weekly,2026-07-08 03:56:30.009200,BD4D080B-1D12-D329-ABD0-39FEB9804E98
101,2026,28,North West,PeakAboveSATCount,9,PeakAboveSATCount_Cadent_Y2026_W28,Weekly,2026-07-08 03:56:30.009200,BD4D080B-1D12-D329-ABD0-39FEB9804E98


In [72]:
from lib.kpi_processor.KPIReport import *
peakSAT = KPIPeakSAT(customer_name)
peakSAT.query_table()
peakSAT.process_data()
peakSAT.data['output']
peakSAT.push_data()

In [78]:
q = Query(query = f"SELECT * FROM KPI_Data")
q.execute(KPIHub_Conn)

,Id,KPIId,CustomerId,BoundaryRegion,Year,PeriodType,PeriodValue,Value,DataType,LastUpdated
0,FOVMain_Cadent_Y2023_W14,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2023,Weekly,14,None,None,2026-07-06 14:26:22.674023
1,FOVMain_Cadent_Y2023_W15,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2023,Weekly,15,None,None,2026-07-06 14:26:22.674023
2,FOVMain_Cadent_Y2023_W16,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2023,Weekly,16,None,None,2026-07-06 14:26:22.674023
3,FOVMain_Cadent_Y2023_W17,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2023,Weekly,17,None,None,2026-07-06 14:26:22.674023
4,FOVMain_Cadent_Y2023_W19,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2023,Weekly,19,None,None,2026-07-06 14:26:22.674023
...,...,...,...,...,...,...,...,...,...,...
30977,PeakAboveSATCount_Cadent_Y2026_W24,PeakAboveSATCount,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2026,Week,24,54,None,2026-07-08 04:09:11.199335
30978,PeakAboveSATCount_Cadent_Y2026_W25,PeakAboveSATCount,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2026,Week,25,52,None,2026-07-08 04:09:11.199335
30979,PeakAboveSATCount_Cadent_Y2026_W26,PeakAboveSATCount,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2026,Week,26,45,None,2026-07-08 04:09:11.199335
30980,PeakAboveSATCount_Cadent_Y2026_W27,PeakAboveSATCount,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2026,Week,27,29,None,2026-07-08 04:09:11.199335
